# 004: Grain Mapping

Written by Jean-Baptiste Jacob

Last updated: 20/02/2026

### Map grains & grain boundaries using misorientation threshold 
- Map grain boundaries as pixels with high misorientation above some threshold
- Do grain segmentation to get the grain mask
- Refine average grain properties using all peaks from the grain masks
- Compute misorientation metrics like Kernel Average Misorentation and deviation from lmean grain orientation (GROD).

All this stuff can be done in MTex as well if you prefer. The export procedure to MTex file (.ctf) is shown at the end of the notebook. 

An alternative grain mapping procedure using density-beased clustering (DBSCAN) in orientation space is detailed in another notebook (`fp4b_Grain_Mapping_DBSCAN`). Works better in 3D or for doing grain tracking in a s3DXRD time series but quite inefficient for large maps. PROBABLY NEEDS UPDATES. TO DO...

### Load packages

In [ ]:
import sys

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess

if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    PYTHONPATH = install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
# general modules
import os
import numpy as np
import matplotlib.pyplot as pl
from tqdm import tqdm

# ImageD11 + point-fit 3dxrd module
import ImageD11.grain
import ImageD11.columnfile
import ImageD11.sinograms.dataset
import ImageD11.friedel_pairs as fp
from pf3dxrd.pf3dxrd import utils, pixelmap, crystal_structure, orientation, peak_mapping

%matplotlib ipympl
%load_ext autoreload
%autoreload 2

### Load data
For now we need only the Pixelmap. We will also need the peakfile later, to refine the average grain orientations once they are segmented. 

In [ ]:
# dataset path
dsfile = '/home/esrf/jean1994b/ES1832/PROCESSED_DATA/MgO_3_0p1M/MgO_3_0p1M_12um_0003/MgO_3_0p1M_12um_0003_dataset.h5'
phase = 'MgO'

In [ ]:
def load_data(dsfile, phase):
    # dataset
    ds = ImageD11.sinograms.dataset.load(dsfile)
    print(ds)
    
    # paths for xmap and peaks
    xmapfile = ds.dsfile.replace('dataset.h5','xmap.h5')
    col2dfile = ds.col2dfile.replace('.h5','_paired.h5')

    # load
    xmap = pixelmap.load_from_hdf5(xmapfile)
    pid = xmap.phases.get(phase).phase_id
    print(xmap)
    
    # load cf  + keep only peaks from the indexed phase
    cf = ImageD11.columnfile.columnfile(col2dfile)
    cf.parameters.loadparameters(ds.parfile)
    fp.update_geometry_fpairs(cf, ds)
    cf.filter(cf.phase_ids==pid)
    utils.get_colf_size(cf)
    print(f'N peaks: {cf.nrows}')
    
    return xmap, cf, ds

In [ ]:
xmap, cf, ds = load_data(dsfile, phase)

In [ ]:
# plot orientation maps to have a look 
for pname in xmap.phases.pnames:
    # skip non-indexed phases
    pm = xmap.get_phase_mask(pname)
    if (pname =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue

    xmap.plot_ipf_orientation(phase=pname, ipf_directions='xyz', show_color_key=True)

### Map grain boundaries and label grains
The function `calcGrains` finds grain boundaries and does grain segmentation based on misorientation threshold. Similar conceptually to MTEX's calcGrains. It computes first the maximal misorientation with pixel neighbors (left-right and top-bottom) over the whole map and defines a grain boundary mask based on an angular threshold specified in input. Grains are then segmented by doing connected component analysis on the binary grain boundary mask. The function works for one phase at a time, so you have to iterate over each indexed phase. 

Grain boundariers are stored separately for each phase in data columns `gb_<phase>` and `gb_misorientation_<phase>`. `grain_id` is updated for each new phase with new grain labels. 

- threshold_deg (float, degree): angle threshold to identify a pixel as a boundary
- min_grain_size (int, pixel nb):  clear out small connected domains with less than N pixels 


In [ ]:
# loop over each phase and mpa grain boundaries
xmap.grain_ids = np.full(xmap.grain_ids.shape, -1)  # initialize grain_id

for phase in xmap.phases.pnames:
    # skip non-indexed phases
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue 
    xmap.calcGrains(phase, Ucol='U', threshold_deg=5, min_grain_size=2)

In [ ]:
xmap.plot('grain_ids')

There is an option for adding grain boundaries masks over xmap plots, by setting `show_gb = True`. This works for every data column, including also the ipf orientation maps. To use it, you must select a phase to plot. There are also two other parameters to control grain boundary aspect:
- `gb_color` : grain boundary color
- `gb_res_fact` : resolution factor; increases the grid size for the grain boundary mask -> makes thinner grain boundaries

If no grain boundary has been computed, it will try to compute them on the fly using default parameters in calcGrains.

In [ ]:
for key in ['grain_ids', 'nindx', 'indx_completeness']:
    xmap.plot(key, phase=phase, show_gb=True, gb_color='r', gb_res_fact=3)    

It is also possible to add multiple grain boundaries (from different 'gb' columns) using the `add_grain_boundaries` method on an existing plot

#### FOR NOISY MAPS: Denoise orientation map

There is a (very basic) denoising function in `pf3dxrd/orientation` to get smoother grain boundaries (can help for grain mapping)

run the following cells for each phase to denoise -> will store smoothed orientation in U_smooth array 

**WARNING!!!** Make sure to write smoothed orientation in separate arrays (otherwise it will overwrite pre-indexed U column)

In [ ]:
phase_to_denoise = 'MgO'  # set here which phase to denoise

cs = xmap.phases.get(phase_to_denoise)
sym = cs.orix_phase.point_group.laue
nx, xb = xmap.grid.nx, xmap.grid.xbins

U = xmap.U.reshape(nx,nx,3,3)
pm = xmap.get_phase_mask(phase_to_denoise)

In [ ]:
U[np.isnan(U)]=0

In [ ]:
# smoothing: compute new orientation array U_smooth. smoothing is done by averaging orientation in the local neighborhood of each pixel.
# kernel_size and threshold_deg control pixel neighbour selection: kernel_size:increases selection window; threshold_deg: excludes neighbors with too large misorientation
U_smooth = orientation.local_orientation_smooth_orix(U, sym, kernel_size=3, threshold_deg=10, global_mask = pm.reshape(nx,nx))

In [ ]:
# segment grains with smoothed data

# add smoothed ori to xmap
if 'U_smooth' not in xmap.titles():
    xmap.add_data(np.zeros_like(xmap.U),'U_smooth')
U_smooth = U_smooth.reshape(nx*nx,3,3)
xmap.update_pixels('U_smooth', U_smooth[pm],selection_mask=pm)

# compute grain boundaries
xmap.calcGrains(phase_to_denoise, Ucol='U_smooth', threshold_deg=2, min_grain_size=3)
xmap.plot_ipf_orientation(phase=phase_to_denoise, datacolname='U_smooth', ipf_directions='xyz', show_gb=True, gb_color='k', gb_res_fact=2)

### map grain_id labels to grains dict
calcGrains updates the `grain_id` column but does not compute average grain properties neither creates new grains. For that we need to use the function `add_grains_from_map`, which will create new grains objects from the grain label map and store them in a dictionnary `xmap.grains.dict`.

In [ ]:
# add grains to xmap.grains dict -> grain obj defined for each segmented domain and added to a dictionnary in xmap.grains
xmap.grains.__init__()

for phase in xmap.phases.pnames:
    
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue 
    
    xmap.add_grains_from_map(phase, Ucol='U', overwrite=False)

print(xmap.grain_ids.max(), max(xmap.grains.gids))    

In [ ]:
xmap.grains.dict.keys()

In [ ]:
# check grain mapping is consistent: same grain labels in xmap.grain_id and grain dict
np.all(np.equal(np.unique(xmap.grain_ids)[1:], np.array(sorted(xmap.grains.gids))))

### Grain refinement
Grain mapping simply averages the unit cell and orientation over each segmented domain, which is a bit dodgy. To refine grains `UBI`, we use the peaks from the peakfile:

1) map grain ids in pixelmap to grain_ids coluln in peakfile based on grain masks
2) for each grain, select peaks from the mask, exclude outliers and merge by unique hkl
3) fit new grain UBI using the merged g-vectors, using a similar procedure as pixel-by-pixel indexing

In [ ]:
# Sanity check: make sure grain labels are all consistent in xmap.grain_id, xmap.grains dict and peakfile
def check_grain_labels(cf, xmap):
    gids_xmap = np.unique(xmap.grain_ids[xmap.phase_ids != -1])
    gids_glist = np.unique(np.array(sorted(xmap.grains.gids)))
    gids_cf = np.unique(cf.grain_id)
    
    assert np.all(np.equal(gids_xmap[1:], gids_glist)), 'grain labels in xmap.grain_id and xmap.grains do not match'
    assert np.all(np.equal(gids_xmap, gids_cf)), 'grain labels in xmap and peakfile do not match'
    
    print('grain labels OK')

In [ ]:
# map peaks to grains
for pid, phase in zip(xmap.phases.pids, xmap.phases.pnames):
    # skip not indexed stuff
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue 
    # first indexed phase (quartz): initialize grains dict
    if pid == 0:
        xmap.map_pks_to_grains(phase, cf, overwrite=True)
    # other phases: update grain dict
    else:
        xmap.map_pks_to_grains(phase, cf, overwrite=False)

In [ ]:
# grain ids have been added to peakfile. We can check unique grain ids in peakfile match those in pixelmap
check_grain_labels(cf, xmap)

Now we can refine each grain `UBI` using all peaks assigned from the grain mask. 

**Notes**
- use large hkl tolerance here, because we want the "average" orientation of grains that can show large intra-granular orientation spread. If hkltol is too small, it will filter out most of the peaks from the grain 
- nmedian is an additional parameter to exclude outliers gvectors plotting too far away from the median of all g-vectors in the selection

In [ ]:
# refine grain ubis based on all peaks mapped to each grains 
stats = {}
for pid, phase in zip(xmap.phases.pids, xmap.phases.pnames):
    # skip not indexed stuff
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue 
    cs = xmap.phases.get(phase)
    stats[phase] = xmap.refine_grain_ubis(phase, cf, hkl_tol=.5, chunksize=10, useInts=True)

`refine_grain_ubis` returns some statistics about refinement for each grain, stored in a dict: 
- `completeness`: similar to `indexing_completeness` in local indexing = fraction of indexed intensity but this time over the whole grain mask
- `mean_drlv2`  : mean deviation from hkl integer for all g-vectors used in the refinement
- `angle deviation` : rotation between older (non-refined) and new orientation matrix, in degree 

### Misorientation
Compute compute misorientation metrics: Kernel average misorientation + Deviation from Mean grain orientation (GROD / Intragranular Misorientation)

In [ ]:
# GROD: Grain Reference Orientation Deviation (see https://mtex-toolbox.github.io/EBSDGROD.html)
# returned as axis-angle orientation per pixel
xmap.calcGROD(pixel_orientation='U')

In [ ]:
# same for each phase in separate plots
for phase in xmap.phases.pnames:
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue
        
    low, up = np.percentile(xmap.GROD_angle[pm & (xmap.GROD_angle>0)], (1,90))
    
    fig = xmap.plot('GROD_angle', phase=phase, show_gb=True, gb_color='k', vmin=low, vmax=up, out=True)

    fig.axes[0].set_title(f'GROD_angle {phase} (deg)')

Grain misorientation axis is also returned as a 3D vector in xyz sample coordinates. This can be used for further misorientation analysis and identification of slip systems (plastically deformed grains)

In [ ]:
xmap.GROD_axis_xyz[xmap.GROD_angle>0]

**Kernel Average Misorientation**

Local Misorientation is computed with the `calcKAM` function. Takes all pixels in local neighborhood and computes average misorientation. This has to be run phase by phase 
- kernel size : controls size of the neighborhood
- threshold_deg: exclude neighbors with too large misorientation (likely from different grains)
- use_numba_acceleration: fast mode to calculate misorientation by simple U matrix comparison, but ignores crystal symmetry
- mode: 'mean' or 'median' misorientation

In [ ]:
# Kernel Average Misorientation
for phase in xmap.phases.pnames:
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue
    xmap.calcKAM(phase, Ucol='U', kernel_size=3, threshold_deg=2, use_numba_acceleration=True, mode='mean')

In [ ]:
for phase in xmap.phases.pnames:
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue
    xmap.plot(f'KAM_{phase}', phase=phase, show_gb=True, gb_color='r', gb_res_fact=2, autoscale=False, norm=pl.matplotlib.colors.LogNorm(vmin=0.01, vmax=0.3))

KAM is computed phase by phase and stored in different columns `KAM_<phase>`. It saves space to merge them into one single array.

In [ ]:
# plot all KAM combined
KAM_merged = np.zeros_like(xmap.KAM_Ni)

for phase in xmap.phases.pnames:
    if f'KAM_{phase}' in xmap.titles():
        pm = xmap.get_phase_mask(phase)
        KAM_merged[pm] = xmap.get(f'KAM_{phase}')[pm]

xmap.add_data(KAM_merged,'KAM')

# delete former KAM_<phase> arrays
for p in xmap.phases.pnames:
    if f'KAM_{p}' in xmap.titles():
        delattr(xmap, f'KAM_{p}')

In [ ]:
fig = xmap.plot('KAM', norm=pl.matplotlib.colors.LogNorm(), out=True)

for i, phase in enumerate(xmap.phases.pnames):
    pm = xmap.get_phase_mask(phase)
    if (phase =='notIndexed') or (xmap.nindx[pm].max() <= 0):
        continue
    xmap.add_grain_boundaries(phase, ax=fig.axes[0], resolution_factor=2, gb_color='r')

### Save refined map and peakfile

In [ ]:
pksfile = os.path.join(ds.analysispath, f'{ds.dsname}_peaks_2d_paired.h5')
utils.colf_to_hdf(cf, pksfile, save_mode='minimal')   

In [ ]:
xmap.h5name = dsfile.replace('dataset.h5','xmap.h5')
xmap.h5name

In [ ]:
xmap.save_to_hdf5()

### Export to MTex .ctf File

There is a built-in method in ImageD11 `TensorMap` that we can use that to create a .ctf file that can be read in Mtex

In [ ]:
# first convert xmap to TensorMap
tmap = xmap.to_tensor_map()

In [ ]:
# xmap puts zeros 3x3 matrices as default 'notIndexed' orientation. Change to np.nan 
m = tmap.nindx <= 0
tmap.U[m] = np.nan

In [ ]:
# then export to ctf
ctf_path = xmap.h5name.replace('.h5','.ctf')
tmap.to_ctf_mtex(ctf_path)

In [ ]:
tmap.to_h5(xmap.h5name.replace('xmap.h5','tmap.h5'))